In [9]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Thiết lập seed để dữ liệu có tính tái lập (reproducible)
np.random.seed(42)

n_rows = 1500  # Số lượng bản ghi telemetry (1500 containers/pallets)

# 1. Tạo định danh cơ bản
container_ids = [f"CONT-{str(i).zfill(5)}" for i in range(1, n_rows + 1)]
sku_categories = np.random.choice(['Standard', 'Oversize', 'Hazmat', 'Fragile'], size=n_rows, p=[0.6, 0.2, 0.1, 0.1])
po_ids = [f"PO-{np.random.randint(10000, 99999)}" for _ in range(n_rows)]

# 2. Tạo mốc thời gian đến bãi (Dock Arrival) trong vòng 7 ngày qua
base_time = datetime(2026, 11, 1, 8, 0, 0)
dock_arrival_times = [
    base_time + timedelta(hours=np.random.randint(0, 168), minutes=np.random.randint(0, 60)) 
    for _ in range(n_rows)
]

# 3. Tạo các biến trạng thái vận hành (State Variables)
# Mô phỏng tình trạng kệ hàng bị đầy nhiều (phân phối lệch về phía cao: 65% - 95%)
bin_utilization = np.random.beta(a=6, b=2, size=n_rows) * 100 

# Quãng đường di chuyển của nhân viên cất hàng (mét)
travel_distance = np.random.normal(loc=55, scale=20, size=n_rows).clip(10, 150)

# Độ trễ hệ thống WMS (ms)
wms_latency = np.random.exponential(scale=40, size=n_rows).astype(int)

# Số lượng nhân sự thực tế trên ca
unloaders_count = np.random.randint(2, 7, size=n_rows)
stowers_count = np.random.randint(4, 14, size=n_rows)

# 4. Tính toán Leadtime Unload-to-Stow (U2S) theo hàm có cấu trúc nhân quả
# Ta thiết lập để Bin Utilization và Travel Distance có trọng số tác động lớn nhất vào U2S
base_processing_time = 25  # phút tối thiểu cố định
congestion_penalty = bin_utilization * 1.8  # Kệ càng đầy, thời gian tăng vọt
travel_penalty = travel_distance * 0.4      # Quãng đường càng xa, thời gian càng lớn
labor_effect = (10 / stowers_count) * 5     # Thiếu nhân sự làm tăng thời gian

# Tổng thời gian U2S (phút) kèm nhiễu ngẫu nhiên (noise)
u2s_minutes = (
    base_processing_time 
    + congestion_penalty 
    + travel_penalty 
    + labor_effect 
    + np.random.normal(0, 8, size=n_rows)
).clip(30, 480).round(2)

# 5. Đóng gói thành DataFrame
df = pd.DataFrame({
    'container_id': container_ids,
    'sku_category': sku_categories,
    'po_id': po_ids,
    'dock_arrival_time': dock_arrival_times,
    'assigned_dock_bay': [f"Bay-{np.random.randint(1, 25)}" for _ in range(n_rows)],
    'bin_utilization_at_stow_%': bin_utilization.round(2),
    'stow_travel_distance_m': travel_distance.round(2),
    'wms_allocation_latency_ms': wms_latency,
    'active_unloaders_count': unloaders_count,
    'active_stowers_count': stowers_count,
    'leadtime_unload_to_stow_mins': u2s_minutes
})

# 6. Xuất ra file Excel
file_name = 'inbound_telemetry_u2s_logs.xlsx'
df.to_excel(file_name, index=False)
print(f"Thành công! Đã tạo file Excel '{file_name}' với {n_rows} dòng dữ liệu telemetry.")

Thành công! Đã tạo file Excel 'inbound_telemetry_u2s_logs.xlsx' với 1500 dòng dữ liệu telemetry.


In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# 1. Đọc dữ liệu từ file Excel đã tạo
file_name = 'inbound_telemetry_u2s_logs.xlsx'
df = pd.read_excel(file_name)

print(f"Đã tải thành công {len(df)} dòng dữ liệu từ {file_name}")

# 2. Định nghĩa các biến đặc trưng (Features) và Biến mục tiêu (Target)
feature_cols = [
    'bin_utilization_at_stow_%', 
    'stow_travel_distance_m', 
    'wms_allocation_latency_ms', 
    'active_unloaders_count', 
    'active_stowers_count'
]
target_col = 'leadtime_unload_to_stow_mins'

X = df[feature_cols]
y = df[target_col]

# 3. Huấn luyện Mô hình Surrogate (Vũ trụ ảo phản ánh quy luật nhà kho)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)
print("Đã huấn luyện xong mô hình Surrogate (Random Forest)!")

# 4. Lọc ra các trường hợp bị nghẽn (Top 20% các lô hàng có U2S cao nhất)
threshold = y.quantile(0.80)
high_u2s_df = df[df[target_col] >= threshold].copy()

print(f"\n--- PHÁT HIỆN ANOMALY ---")
print( ngưỡng := f"Ngưỡng U2S cao (>= Top 20%): {threshold:.2f} phút")
print(f"Số lượng bản ghi bị nghẽn cần phân tích RCA: {len(high_u2s_df)} bản ghi")

# 5. Định nghĩa các giá trị "Lý tưởng" (Optimal Counterfactual Values) cho từng yếu tố
ideal_values = {
    'bin_utilization_at_stow_%': 45.0,       # Giảm tải kệ về 45% (thoáng)
    'stow_travel_distance_m': 30.0,          # Rút ngắn quãng đường di chuyển
    'wms_allocation_latency_ms': 5.0,        # Giảm trễ WMS gần như 0
    'active_unloaders_count': 6,             # Tối đa nhân lực dỡ hàng
    'active_stowers_count': 12               # Tối đa nhân lực cất hàng
}

# 6. Động cơ Can thiệp Phản thực (Counterfactual Intervention Engine)
baseline_preds = model.predict(high_u2s_df[feature_cols])
mean_baseline_u2s = baseline_preds.mean()

impact_scores = {}

for feature in feature_cols:
    # Tạo thế giới phản thực (copy dữ liệu nghẽn)
    cf_data = high_u2s_df[feature_cols].copy()
    
    # Can thiệp duy nhất (do-calculus): Đưa biến này về giá trị lý tưởng, giữ nguyên các biến khác
    cf_data[feature] = ideal_values[feature]
    
    # Dự đoán U2S trong thế giới phản thực đó
    cf_preds = model.predict(cf_data)
    mean_cf_u2s = cf_preds.mean()
    
    # Tính điểm tác động: U2S giảm đi được bao nhiêu phút?
    impact_score = mean_baseline_u2s - mean_cf_u2s
    impact_scores[feature] = impact_score

# 7. Xếp hạng Nguyên nhân Gốc rễ (Root Cause Ranking)
print("\n=== KẾT QUẢ PHÂN TÍCH NGUYÊN NHÂN GỐC RỄ (RCA) PHẢN THỰC ===")
sorted_impacts = sorted(impact_scores.items(), key=lambda x: x[1], reverse=True)

for rank, (feat, score) in enumerate(sorted_impacts, 1):
    print(f"Hạng {rank}: {feat:<30} | Tác động giảm U2S: {score:.2f} phút")

top_root_cause = sorted_impacts[0][0]
print(f"\n=> KẾT LUẬN TỪ CONTROL TOWER: Nguyên nhân gốc rễ lớn nhất gây chậm U2S là [{top_root_cause}]!")

Đã tải thành công 1500 dòng dữ liệu từ inbound_telemetry_u2s_logs.xlsx
Đã huấn luyện xong mô hình Surrogate (Random Forest)!

--- PHÁT HIỆN ANOMALY ---
Ngưỡng U2S cao (>= Top 20%): 213.70 phút
Số lượng bản ghi bị nghẽn cần phân tích RCA: 300 bản ghi

=== KẾT QUẢ PHÂN TÍCH NGUYÊN NHÂN GỐC RỄ (RCA) PHẢN THỰC ===
Hạng 1: bin_utilization_at_stow_%      | Tác động giảm U2S: 87.04 phút
Hạng 2: stow_travel_distance_m         | Tác động giảm U2S: 14.46 phút
Hạng 3: active_stowers_count           | Tác động giảm U2S: 2.92 phút
Hạng 4: wms_allocation_latency_ms      | Tác động giảm U2S: 0.42 phút
Hạng 5: active_unloaders_count         | Tác động giảm U2S: 0.38 phút

=> KẾT LUẬN TỪ CONTROL TOWER: Nguyên nhân gốc rễ lớn nhất gây chậm U2S là [bin_utilization_at_stow_%]!
